# Demand Forecasting & Inventory Planning for a Packaging Distributor
**Goal:** forecast monthly demand for 15 packaging SKUs, choose the most accurate
forecasting method for each SKU, and turn the forecast into an inventory plan
(safety stock + reorder point) at a 95% service level.

**Steps**
1. Load and explore 36 months of sales history
2. Build 5 forecasting methods
3. Pick the best method per SKU (validation) and measure accuracy honestly (test)
4. Forecast the next 6 months
5. Calculate safety stock and reorder points
6. Save charts and results

Only uses pandas, numpy and matplotlib (all pre-installed in Google Colab).

**How to run:** In Google Colab click *Runtime > Run all*. Everything (data, charts, Excel file) is created automatically. Download the results from the folder icon on the left (`outputs/`).

## Step 0 - Create the simulated sales dataset
36 months of monthly unit sales for 15 packaging SKUs, with growth trend, a Q4 holiday-shipping peak and random noise. Data is simulated, not real company data.

In [ ]:
import os
os.makedirs("data", exist_ok=True)
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # fixed seed -> same data every run

# sku, description, category, base monthly units, annual growth, seasonal strength, noise, unit cost $, lead time (weeks)
SKUS = [
    ("BOX-1210", "Corrugated Box 12x10x8",      "Corrugated Boxes", 5200, 0.08, 0.35, 0.08, 0.62, 3),
    ("BOX-1812", "Corrugated Box 18x12x12",     "Corrugated Boxes", 3100, 0.06, 0.30, 0.09, 1.05, 3),
    ("BOX-2420", "Heavy-Duty Box 24x20x20",     "Corrugated Boxes", 1400, 0.04, 0.25, 0.12, 2.40, 4),
    ("TAPE-2CL", "Packing Tape 2in Clear",      "Tape",             4600, 0.05, 0.30, 0.07, 1.15, 2),
    ("TAPE-3BR", "Carton Tape 3in Brown",       "Tape",             1800, 0.03, 0.20, 0.10, 1.60, 2),
    ("BUB-1224", "Bubble Roll 12in x 250ft",    "Protective Pkg",    950, 0.07, 0.40, 0.11, 18.50, 5),
    ("BUB-PCH",  "Bubble Pouch 6x8 (case)",     "Protective Pkg",    700, 0.10, 0.35, 0.13, 22.00, 5),
    ("FILM-18",  "Stretch Film 18in x 1500ft",  "Stretch Film",     2300, 0.02, 0.10, 0.08, 14.75, 4),
    ("FILM-HD",  "Hand Stretch Film 80ga",      "Stretch Film",      850, 0.01, 0.10, 0.12, 19.90, 4),
    ("MAIL-1013","Poly Mailer 10x13",           "Mailers",          6800, 0.12, 0.45, 0.09, 0.14, 3),
    ("MAIL-BUB", "Bubble Mailer #2",            "Mailers",          3900, 0.09, 0.40, 0.10, 0.32, 3),
    ("LBL-4X6",  "Thermal Shipping Label 4x6",  "Labels",           2600, 0.11, 0.30, 0.08, 9.80, 2),
    ("VOID-AIR", "Air Pillow Void Fill (case)", "Protective Pkg",    600, 0.05, 0.35, 0.15, 26.00, 6),
    ("PAL-4840", "Plastic Pallet 48x40",        "Warehouse",         180, 0.02, 0.05, 0.18, 89.00, 8),
    ("GLV-NIT",  "Nitrile Gloves (case)",       "Warehouse",         520, 0.00, 0.05, 0.14, 34.00, 6),
]

# Monthly seasonal shape for a shipping-supplies business:
# quiet Jan-Feb, steady mid-year, build-up in Sep-Oct, peak Nov-Dec (holiday shipping)
SEASON_SHAPE = np.array([-0.55, -0.45, -0.15, -0.10, -0.05, -0.10,
                         -0.15, -0.05,  0.15,  0.35,  0.55,  0.55])

dates = pd.date_range("2023-01-01", "2025-12-01", freq="MS")
rows = []
for sku, desc, cat, base, growth, seas, noise, cost, lt in SKUS:
    for t, d in enumerate(dates):
        trend = (1 + growth) ** (t / 12)
        season = 1 + seas * SEASON_SHAPE[d.month - 1]
        shock = rng.normal(1, noise)
        units = max(0, round(base * trend * season * shock))
        rows.append({"month": d.strftime("%Y-%m"), "sku": sku, "description": desc,
                     "category": cat, "units_sold": units, "unit_cost": cost,
                     "lead_time_weeks": lt})

df = pd.DataFrame(rows)
df.to_csv("data/packaging_sales_history.csv", index=False)
print(f"Saved {len(df)} rows ({df.sku.nunique()} SKUs x {df.month.nunique()} months) "
      f"to data/packaging_sales_history.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("outputs", exist_ok=True)
df = pd.read_csv("data/packaging_sales_history.csv")
df["month"] = pd.to_datetime(df["month"])
print(df.head())
print(f"\n{df.sku.nunique()} SKUs, {df.month.nunique()} months: "
      f"{df.month.min():%b %Y} to {df.month.max():%b %Y}")

## Step 1 - Explore the data
Look for **trend** (growing over time?) and **seasonality** (repeating yearly pattern?).

In [ ]:
total = df.groupby("month")["units_sold"].sum()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(total.index, total.values, marker="o", color="#1f4e79")
ax.set_title("Total Monthly Units Sold - All 15 SKUs")
ax.set_ylabel("Units")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/01_total_sales_trend.png", dpi=150)
plt.show()

by_month = df.assign(m=df.month.dt.month).groupby("m")["units_sold"].sum()
print("Share of annual volume by calendar month (seasonality):")
print((by_month / by_month.sum()).round(3).to_string())

yearly = df.groupby(df.month.dt.year)["units_sold"].sum()
print("\nYear-over-year growth:", (yearly.pct_change() * 100).round(1).dropna().to_dict())

## Step 2 - Five forecasting methods
| Method | Idea |
|---|---|
| Naive | Next month = this month (the baseline every model must beat) |
| 3-Month Moving Average | Average of the last 3 months |
| Seasonal Naive | Next July = last July |
| Simple Exponential Smoothing | Weighted average, recent months count more |
| Holt-Winters | Smoothing that captures level + trend + seasonality |

Each function takes the history (`y`) and returns forecasts for the next `h` months.

In [ ]:
SEASON = 12

def naive(y, h):
    return np.repeat(y[-1], h)

def moving_average(y, h, window=3):
    return np.repeat(y[-window:].mean(), h)

def seasonal_naive(y, h):
    return np.array([y[len(y) - SEASON + (i % SEASON)] for i in range(h)])

def ses(y, h, alpha):
    level = y[0]
    for value in y[1:]:
        level = alpha * value + (1 - alpha) * level
    return np.repeat(level, h)

def holt_winters(y, h, alpha, beta, gamma):
    """Additive Holt-Winters. Needs at least 2 years of history."""
    level = y[:SEASON].mean()
    trend = (y[SEASON:2 * SEASON].mean() - y[:SEASON].mean()) / SEASON
    season = list(y[:SEASON] - level)
    for t in range(len(y)):
        s = season[t]
        last_level = level
        level = alpha * (y[t] - s) + (1 - alpha) * (level + trend)
        trend = beta * (level - last_level) + (1 - beta) * trend
        season.append(gamma * (y[t] - level) + (1 - gamma) * s)
    n = len(y)
    # season[n + i] holds the latest seasonal estimate for future month n + i
    return np.array([level + (i + 1) * trend + season[n + (i % SEASON)] for i in range(h)])

def one_step_error(y, model, params):
    """Average one-month-ahead error inside the history - used to tune parameters.
    Walks through the history once: predict month t, compare with actual, update."""
    errs = []
    if model is ses:
        (alpha,) = params
        level = y[0]
        for t in range(1, len(y)):
            errs.append(abs(y[t] - level))
            level = alpha * y[t] + (1 - alpha) * level
    else:
        alpha, beta, gamma = params
        level = y[:SEASON].mean()
        trend = (y[SEASON:2 * SEASON].mean() - y[:SEASON].mean()) / SEASON
        season = list(y[:SEASON] - level)
        for t in range(len(y)):
            if t >= SEASON:  # skip year 1 (used to set starting values)
                errs.append(abs(y[t] - (level + trend + season[t])))
            last_level = level
            level = alpha * (y[t] - season[t]) + (1 - alpha) * (level + trend)
            trend = beta * (level - last_level) + (1 - beta) * trend
            season.append(gamma * (y[t] - level) + (1 - gamma) * season[t])
    return np.mean(errs)

def tune(y, model):
    """Try a grid of smoothing parameters and keep the one with the lowest error."""
    if model is ses:
        grid = [(a,) for a in np.arange(0.1, 1.0, 0.1)]
    else:
        grid = [(a, b, g) for a in (0.1, 0.2, 0.3, 0.5)
                for b in (0.01, 0.05, 0.1) for g in (0.1, 0.3, 0.5)]
    return min(grid, key=lambda p: one_step_error(y, model, p))

def forecast(method, y, h):
    if method == "Naive":
        return naive(y, h)
    if method == "3-Month Moving Avg":
        return moving_average(y, h)
    if method == "Seasonal Naive":
        return seasonal_naive(y, h)
    if method == "Exp. Smoothing":
        return ses(y, h, *tune(y, ses))
    if method == "Holt-Winters":
        return holt_winters(y, h, *tune(y, holt_winters))

METHODS = ["Naive", "3-Month Moving Avg", "Seasonal Naive", "Exp. Smoothing", "Holt-Winters"]

## Step 3 - Validation and test (honest accuracy)
* **Validation:** train on months 1-24, forecast months 25-30 -> pick the best method per SKU.
* **Test:** train on months 1-30, forecast months 31-36 (Jul-Dec 2025, includes the holiday peak)
  using the method chosen in validation -> this is the accuracy we report.

Picking the method on one period and scoring it on a *different* period stops us
from "cheating" by choosing whatever happened to look best on the test data.

**MAPE** = average of |actual - forecast| / actual. Lower is better.
**WAPE** = total |error| / total actual units (portfolio-level accuracy, volume-weighted).

In [ ]:
H = 6
skus = df.sku.unique()
series = {s: df[df.sku == s].sort_values("month")["units_sold"].to_numpy(float) for s in skus}
months = sorted(df.month.unique())

def mape(actual, fc):
    return np.mean(np.abs(actual - fc) / actual)

val_rows, test_rows = [], []
for s in skus:
    y = series[s]
    for m in METHODS:
        fc = forecast(m, y[:24], H)
        val_rows.append({"sku": s, "method": m, "val_mape": mape(y[24:30], fc)})
        fc_test = forecast(m, y[:30], H)
        for i in range(H):
            test_rows.append({"sku": s, "method": m, "month": months[30 + i],
                              "actual": y[30 + i], "forecast": round(fc_test[i])})

val = pd.DataFrame(val_rows)
test = pd.DataFrame(test_rows)
test["abs_error"] = (test.actual - test.forecast).abs()
test["ape"] = test.abs_error / test.actual

best = val.loc[val.groupby("sku")["val_mape"].idxmin(), ["sku", "method"]]
best = best.rename(columns={"method": "best_method"}).set_index("sku")["best_method"]
print("Best method per SKU (chosen on validation):")
print(best.value_counts().to_string())

In [ ]:
summary = test.groupby("method").agg(MAPE=("ape", "mean"),
                                     abs_err=("abs_error", "sum"),
                                     actual=("actual", "sum"))
summary["WAPE"] = summary.abs_err / summary.actual
summary = summary[["MAPE", "WAPE"]].reindex(METHODS)

chosen = test[test.method == test.sku.map(best)]
summary.loc["Best-per-SKU (selected)"] = [chosen.ape.mean(),
                                          chosen.abs_error.sum() / chosen.actual.sum()]
print("Test accuracy, Jul-Dec 2025:")
print((summary * 100).round(1).astype(str).add("%").to_string())

naive_wape = summary.loc["Naive", "WAPE"]
best_wape = summary.loc["Best-per-SKU (selected)", "WAPE"]
improvement = (naive_wape - best_wape) / naive_wape
print(f"\nForecast error reduced by {improvement:.0%} vs. the naive baseline "
      f"(WAPE {naive_wape:.1%} -> {best_wape:.1%}); "
      f"forecast accuracy = {1 - best_wape:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = summary["WAPE"] * 100
colors = ["#9dc3e6"] * (len(plot_df) - 1) + ["#1f4e79"]
ax.barh(plot_df.index, plot_df.values, color=colors)
for i, v in enumerate(plot_df.values):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center")
ax.invert_yaxis()
ax.set_xlim(0, plot_df.max() * 1.15)
ax.set_xlabel("Forecast error, WAPE % (lower is better)")
ax.set_title("Forecast Error by Method - Test Period Jul-Dec 2025")
plt.tight_layout()
plt.savefig("outputs/02_model_comparison.png", dpi=150)
plt.show()

tot = chosen.groupby("month")[["actual", "forecast"]].sum()
naive_tot = test[test.method == "Naive"].groupby("month")["forecast"].sum()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(total.index, total.values, color="#7f7f7f", label="Actual units sold")
ax.plot(tot.index, tot.forecast, "o--", color="#1f4e79", label="Selected model forecast")
ax.plot(naive_tot.index, naive_tot.values, "x:", color="#c55a11", label="Naive baseline")
ax.set_title("Test Period: Forecast vs Actual (all SKUs)")
ax.set_ylabel("Units")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/03_forecast_vs_actual.png", dpi=150)
plt.show()

## Step 4 - Forecast the next 6 months (Jan-Jun 2026)
Re-train each SKU's chosen method on all 36 months.

In [ ]:
future_months = pd.date_range(months[-1] + pd.offsets.MonthBegin(1), periods=H, freq="MS")
fc_rows = []
for s in skus:
    fc = forecast(best[s], series[s], H)
    for i, m in enumerate(future_months):
        fc_rows.append({"sku": s, "month": m, "forecast_units": max(0, round(fc[i]))})
future = pd.DataFrame(fc_rows)
print(future.pivot(index="sku", columns="month", values="forecast_units")
      .rename(columns=lambda c: c.strftime("%b %Y")).to_string())

## Step 5 - Inventory plan: safety stock & reorder point
* **Safety stock** = Z x (forecast error, measured as RMSE) x sqrt(lead time in months)
  - Z = 1.645 for a 95% service level (stock out in only ~5% of cycles)
* **Reorder point** = average monthly demand x lead time (months) + safety stock
  - When on-hand + on-order inventory drops to this level, place a new order.

In [ ]:
Z = 1.645
info = df.groupby("sku")[["description", "category", "unit_cost", "lead_time_weeks"]].first()
plan = info.copy()
plan["method"] = best
plan["avg_monthly_fc"] = future.groupby("sku")["forecast_units"].mean()
def rmse(frame):
    """Root mean squared forecast error per SKU - the typical size of a monthly miss."""
    e = frame.actual - frame.forecast
    return np.sqrt((e ** 2).groupby(frame.sku).mean())

plan["forecast_error_sd"] = rmse(chosen)
plan["lead_time_months"] = plan.lead_time_weeks / 4.33
plan["safety_stock"] = np.ceil(Z * plan.forecast_error_sd * np.sqrt(plan.lead_time_months))
plan["reorder_point"] = np.ceil(plan.avg_monthly_fc * plan.lead_time_months + plan.safety_stock)
plan["safety_stock_value"] = (plan.safety_stock * plan.unit_cost).round(2)
print(plan[["method", "avg_monthly_fc", "safety_stock", "reorder_point", "safety_stock_value"]]
      .round(0).to_string())
print(f"\nTotal safety stock investment: ${plan.safety_stock_value.sum():,.0f}")

# Better forecasts = smaller errors = less safety stock needed for the same 95% service level.
naive_test = test[test.method == "Naive"]
naive_sd = rmse(naive_test)
plan["safety_stock_if_naive"] = np.ceil(Z * naive_sd * np.sqrt(plan.lead_time_months))
naive_ss_value = (plan.safety_stock_if_naive * plan.unit_cost).sum()
ss_saving = naive_ss_value - plan.safety_stock_value.sum()
print(f"Safety stock needed with the naive forecast: ${naive_ss_value:,.0f}")
print(f"Savings from better forecasting: ${ss_saving:,.0f} "
      f"({ss_saving / naive_ss_value:.0%} less inventory at the same 95% service level)")

## Step 6 - Save results
CSV files feed the Excel dashboard (`build_excel.py`).

In [ ]:
test.to_csv("outputs/backtest_results.csv", index=False)
val.to_csv("outputs/validation_results.csv", index=False)
future.to_csv("outputs/forecast_next_6_months.csv", index=False)
plan.reset_index().to_csv("outputs/inventory_plan.csv", index=False)
best.reset_index().to_csv("outputs/best_method_per_sku.csv", index=False)
summary.reset_index().rename(columns={"index": "method"}).to_csv(
    "outputs/model_summary.csv", index=False)
print("Saved all results to outputs/")

## Step 7 - Build the Excel dashboard
Creates `outputs/Demand_Forecasting_Inventory_Plan.xlsx` with live formulas (change the service level in *Inventory Plan!B3*). Excel calculates the formulas when you open the file.

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

FONT = "Arial"
NAVY = "1F4E79"
HDR_FILL = PatternFill("solid", fgColor=NAVY)
INPUT_FILL = PatternFill("solid", fgColor="FFFF00")
KPI_FILL = PatternFill("solid", fgColor="DDEBF7")
THIN = Side(style="thin", color="BFBFBF")
BOX = Border(top=THIN, bottom=THIN, left=THIN, right=THIN)

f = lambda **k: Font(name=FONT, **k)
BLUE = f(color="0000FF")   # hardcoded inputs / model outputs pasted from Python
BLACK = f()

hist = pd.read_csv("data/packaging_sales_history.csv")
test = pd.read_csv("outputs/backtest_results.csv", parse_dates=["month"])
future = pd.read_csv("outputs/forecast_next_6_months.csv", parse_dates=["month"])
plan = pd.read_csv("outputs/inventory_plan.csv")
METHODS = ["Naive", "3-Month Moving Avg", "Seasonal Naive", "Exp. Smoothing", "Holt-Winters"]
skus = list(plan.sku)

wb = Workbook()
dash = wb.active
dash.title = "Dashboard"
readme = wb.create_sheet("Read Me")
inv = wb.create_sheet("Inventory Plan")
fcs = wb.create_sheet("Forecast 2026")
mc = wb.create_sheet("Model Comparison")
bt = wb.create_sheet("Backtest")
wide = wb.create_sheet("Sales by SKU")
sh = wb.create_sheet("Sales History")


def header(ws, row, labels, col=1):
    for i, lab in enumerate(labels):
        c = ws.cell(row=row, column=col + i, value=lab)
        c.font = f(bold=True, color="FFFFFF")
        c.fill = HDR_FILL
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border = BOX


def widths(ws, w):
    for i, width in enumerate(w, start=1):
        ws.column_dimensions[get_column_letter(i)].width = width


def title(ws, text, sub=None):
    ws["A1"] = text
    ws["A1"].font = f(bold=True, size=14, color=NAVY)
    if sub:
        ws["A2"] = sub
        ws["A2"].font = f(italic=True, color="595959")


# ---------------- Sales History (raw data) ----------------
title(sh, "Sales History - simulated monthly units, Jan 2023 to Dec 2025",
      "Source: data/packaging_sales_history.csv (simulated data generated by generate_data.py)")
cols = ["Month", "SKU", "Description", "Category", "Units Sold", "Unit Cost ($)", "Lead Time (wks)"]
header(sh, 4, cols)
for r, row in enumerate(hist.itertuples(index=False), start=5):
    vals = [pd.Timestamp(row.month + "-01"), row.sku, row.description, row.category,
            row.units_sold, row.unit_cost, row.lead_time_weeks]
    for c, v in enumerate(vals, start=1):
        cell = sh.cell(row=r, column=c, value=v)
        cell.font = BLUE if c in (5, 6, 7) else BLACK
    sh.cell(row=r, column=1).number_format = "mmm yyyy"
    sh.cell(row=r, column=5).number_format = "#,##0"
    sh.cell(row=r, column=6).number_format = "$#,##0.00"
SH_LAST = 4 + len(hist)
widths(sh, [11, 12, 30, 18, 12, 13, 15])
sh.freeze_panes = "A5"

# ---------------- Sales by SKU (SUMIFS matrix) ----------------
title(wide, "Monthly Units by SKU (built with SUMIFS from Sales History)")
months = sorted(pd.to_datetime(hist.month + "-01").unique())
header(wide, 4, ["SKU"] + [None] * len(months) + ["Total", "Avg / Month"])
for j, m in enumerate(months, start=2):
    c = wide.cell(row=4, column=j, value=pd.Timestamp(m))
    c.number_format = "mmm yy"
for i, s in enumerate(skus, start=5):
    wide.cell(row=i, column=1, value=s).font = BLACK
    for j in range(2, len(months) + 2):
        col = get_column_letter(j)
        c = wide.cell(row=i, column=j,
                      value=f"=SUMIFS('Sales History'!$E$5:$E${SH_LAST},"
                            f"'Sales History'!$B$5:$B${SH_LAST},$A{i},"
                            f"'Sales History'!$A$5:$A${SH_LAST},{col}$4)")
        c.number_format = "#,##0"
    last = get_column_letter(len(months) + 1)
    wide.cell(row=i, column=len(months) + 2, value=f"=SUM(B{i}:{last}{i})").number_format = "#,##0"
    wide.cell(row=i, column=len(months) + 3,
              value=f"=AVERAGE(B{i}:{last}{i})").number_format = "#,##0"
tr = 5 + len(skus)
wide.cell(row=tr, column=1, value="Total").font = f(bold=True)
for j in range(2, len(months) + 4):
    col = get_column_letter(j)
    c = wide.cell(row=tr, column=j, value=f"=SUM({col}5:{col}{tr - 1})")
    c.number_format = "#,##0"
    c.font = f(bold=True)
widths(wide, [12] + [8] * len(months) + [11, 11])
wide.freeze_panes = "B5"

# ---------------- Backtest ----------------
title(bt, "Backtest - Jul to Dec 2025 (models trained on Jan 2023 - Jun 2025)",
      "Blue = values from Python. Black = Excel formulas. Selected = method chosen for that SKU "
      "on the separate validation period.")
header(bt, 4, ["SKU", "Method", "Month", "Actual Units", "Forecast Units",
               "Abs Error", "Abs % Error", "Selected?"])
for r, row in enumerate(test.itertuples(index=False), start=5):
    bt.cell(row=r, column=1, value=row.sku)
    bt.cell(row=r, column=2, value=row.method)
    c = bt.cell(row=r, column=3, value=row.month.to_pydatetime())
    c.number_format = "mmm yyyy"
    for col, v in ((4, row.actual), (5, row.forecast)):
        c = bt.cell(row=r, column=col, value=float(v))
        c.font = BLUE
        c.number_format = "#,##0"
    bt.cell(row=r, column=6, value=f"=ABS(D{r}-E{r})").number_format = "#,##0"
    bt.cell(row=r, column=7, value=f"=IF(D{r}=0,0,F{r}/D{r})").number_format = "0.0%"
    bt.cell(row=r, column=8,
            value=f"=IF(B{r}=INDEX('Inventory Plan'!$D$7:$D${6 + len(skus)},"
                  f"MATCH(A{r},'Inventory Plan'!$A$7:$A${6 + len(skus)},0)),\"Yes\",\"No\")")
BT_LAST = 4 + len(test)
widths(bt, [12, 20, 11, 13, 14, 11, 12, 11])
bt.freeze_panes = "A5"
bt.auto_filter.ref = f"A4:H{BT_LAST}"

rng = lambda col: f"Backtest!${col}$5:${col}${BT_LAST}"

# ---------------- Model Comparison ----------------
title(mc, "Model Comparison - forecast accuracy on the Jul-Dec 2025 test period",
      "MAPE = average % miss per SKU-month.  WAPE = total units missed / total units sold "
      "(volume-weighted; the headline metric).")
header(mc, 4, ["Method", "MAPE", "WAPE", "Forecast Accuracy (1 - WAPE)", "Error Reduction vs Naive"])
for i, m in enumerate(METHODS + ["Best-per-SKU (selected)"], start=5):
    mc.cell(row=i, column=1, value=m)
    if m.startswith("Best"):
        mc.cell(row=i, column=2, value=f'=AVERAGEIFS({rng("G")},{rng("H")},"Yes")')
        mc.cell(row=i, column=3,
                value=f'=SUMIFS({rng("F")},{rng("H")},"Yes")/SUMIFS({rng("D")},{rng("H")},"Yes")')
        for c in range(1, 6):
            mc.cell(row=i, column=c).font = f(bold=True)
            mc.cell(row=i, column=c).fill = KPI_FILL
    else:
        mc.cell(row=i, column=2, value=f"=AVERAGEIFS({rng('G')},{rng('B')},A{i})")
        mc.cell(row=i, column=3,
                value=f"=SUMIFS({rng('F')},{rng('B')},A{i})/SUMIFS({rng('D')},{rng('B')},A{i})")
    mc.cell(row=i, column=4, value=f"=1-C{i}")
    mc.cell(row=i, column=5, value=f"=($C$5-C{i})/$C$5")
    for c in range(2, 6):
        mc.cell(row=i, column=c).number_format = "0.0%"
    for c in range(1, 6):
        mc.cell(row=i, column=c).border = BOX
MC_SEL = 10
mc["A12"] = "Selected methods by SKU"
mc["A12"].font = f(bold=True, color=NAVY)
header(mc, 13, ["Method", "# of SKUs"])
for i, m in enumerate(METHODS, start=14):
    mc.cell(row=i, column=1, value=m)
    mc.cell(row=i, column=2, value=f"=COUNTIF('Inventory Plan'!$D$7:$D${6 + len(skus)},A{i})")
widths(mc, [26, 11, 11, 16, 16])

bar = BarChart()
bar.type = "bar"
bar.title = "Forecast Error (WAPE) by Method - lower is better"
bar.y_axis.title = "WAPE"
bar.y_axis.numFmt = "0%"
bar.add_data(Reference(mc, min_col=3, min_row=4, max_row=MC_SEL), titles_from_data=True)
bar.set_categories(Reference(mc, min_col=1, min_row=5, max_row=MC_SEL))
bar.x_axis.scaling.orientation = "maxMin"
bar.legend = None
bar.height, bar.width = 7.5, 16

# ---------------- Forecast 2026 ----------------
title(fcs, "6-Month Demand Forecast, Jan - Jun 2026 (units)",
      "Blue = forecast produced in Python by each SKU's selected method, re-trained on all 36 months.")
fmonths = sorted(future.month.unique())
header(fcs, 4, ["SKU", "Description", "Method"] + [None] * len(fmonths) + ["6-Mo Total", "Avg / Month"])
for j, m in enumerate(fmonths, start=4):
    c = fcs.cell(row=4, column=j, value=pd.Timestamp(m).to_pydatetime())
    c.number_format = "mmm yyyy"
piv = future.pivot(index="sku", columns="month", values="forecast_units")
for i, s in enumerate(skus, start=5):
    p = plan.set_index("sku").loc[s]
    fcs.cell(row=i, column=1, value=s)
    fcs.cell(row=i, column=2, value=p.description)
    fcs.cell(row=i, column=3, value=f"=INDEX('Inventory Plan'!$D$7:$D${6 + len(skus)},"
                                    f"MATCH(A{i},'Inventory Plan'!$A$7:$A${6 + len(skus)},0))")
    for j, m in enumerate(fmonths, start=4):
        c = fcs.cell(row=i, column=j, value=int(piv.loc[s, m]))
        c.font = BLUE
        c.number_format = "#,##0"
    a, b = get_column_letter(4), get_column_letter(3 + len(fmonths))
    fcs.cell(row=i, column=4 + len(fmonths), value=f"=SUM({a}{i}:{b}{i})").number_format = "#,##0"
    fcs.cell(row=i, column=5 + len(fmonths), value=f"=AVERAGE({a}{i}:{b}{i})").number_format = "#,##0"
FC_LAST = 4 + len(skus)
tr = FC_LAST + 1
fcs.cell(row=tr, column=1, value="Total").font = f(bold=True)
for j in range(4, 6 + len(fmonths)):
    col = get_column_letter(j)
    c = fcs.cell(row=tr, column=j, value=f"=SUM({col}5:{col}{FC_LAST})")
    c.number_format = "#,##0"
    c.font = f(bold=True)
widths(fcs, [12, 30, 18] + [11] * len(fmonths) + [12, 12])
fcs.freeze_panes = "D5"

# ---------------- Inventory Plan ----------------
title(inv, "Inventory Plan - Safety Stock & Reorder Points")
inv["A3"] = "Target service level"
inv["B3"] = 0.95
inv["B3"].fill = INPUT_FILL
inv["B3"].font = BLUE
inv["B3"].number_format = "0.0%"
inv["C3"] = "<- change this (e.g. 90%, 98%) and the whole plan recalculates"
inv["C3"].font = f(italic=True, color="595959")
inv["A4"] = "Z-score"
inv["B4"] = "=NORMSINV(B3)"
inv["B4"].number_format = "0.000"
inv["A3"].font = inv["A4"].font = f(bold=True)
inv["B3"].comment = Comment("Input. 95% = expect to cover demand without a stockout in ~95% of "
                            "replenishment cycles.", "Model")
cols = ["SKU", "Description", "Category", "Selected Method", "Unit Cost ($)", "Lead Time (wks)",
        "Lead Time (months)", "Avg Monthly Forecast", "Forecast Error RMSE (units)",
        "Safety Stock (units)", "Reorder Point (units)", "Safety Stock Value ($)",
        "Naive Error RMSE (units)", "Safety Stock if Naive (units)", "Safety Stock Value if Naive ($)"]
header(inv, 6, cols)
inv.row_dimensions[6].height = 45
P0, P1 = 7, 6 + len(skus)
for i, row in enumerate(plan.itertuples(index=False), start=P0):
    fr = 5 + skus.index(row.sku)
    vals = {
        1: row.sku, 2: row.description, 3: row.category, 4: row.method,
        5: row.unit_cost, 6: int(row.lead_time_weeks),
        7: f"=F{i}/4.33",
        8: f"='Forecast 2026'!{get_column_letter(5 + len(fmonths))}{fr}",
        9: f'=SQRT(SUMPRODUCT(({rng("A")}=A{i})*({rng("H")}="Yes")*({rng("D")}-{rng("E")})^2)'
           f'/COUNTIFS({rng("A")},A{i},{rng("H")},"Yes"))',
        10: f"=ROUNDUP($B$4*I{i}*SQRT(G{i}),0)",
        11: f"=ROUNDUP(H{i}*G{i}+J{i},0)",
        12: f"=J{i}*E{i}",
        13: f'=SQRT(SUMPRODUCT(({rng("A")}=A{i})*({rng("B")}="Naive")*({rng("D")}-{rng("E")})^2)'
            f'/COUNTIFS({rng("A")},A{i},{rng("B")},"Naive"))',
        14: f"=ROUNDUP($B$4*M{i}*SQRT(G{i}),0)",
        15: f"=N{i}*E{i}",
    }
    for c, v in vals.items():
        cell = inv.cell(row=i, column=c, value=v)
        cell.border = BOX
        if c in (4, 5, 6):
            cell.font = BLUE
    inv.cell(row=i, column=4).comment = None
    for c, fmt in ((5, "$#,##0.00"), (7, "0.00"), (8, "#,##0"), (9, "#,##0"), (10, "#,##0"),
                   (11, "#,##0"), (12, "$#,##0"), (13, "#,##0"), (14, "#,##0"), (15, "$#,##0")):
        inv.cell(row=i, column=c).number_format = fmt
inv.cell(row=P0, column=4).comment = Comment(
    "Method chosen per SKU in Python using the Jan-Jun 2025 validation period.", "Model")
tr = P1 + 1
inv.cell(row=tr, column=1, value="Total").font = f(bold=True)
for c in (12, 15):
    col = get_column_letter(c)
    cell = inv.cell(row=tr, column=c, value=f"=SUM({col}{P0}:{col}{P1})")
    cell.number_format = "$#,##0"
    cell.font = f(bold=True)
inv.cell(row=tr + 2, column=1, value="Safety stock saved vs. naive forecasting").font = f(bold=True)
inv.cell(row=tr + 2, column=5, value=f"=O{tr}-L{tr}").number_format = "$#,##0"
inv.cell(row=tr + 3, column=1, value="Reduction (%)").font = f(bold=True)
inv.cell(row=tr + 3, column=5, value=f"=IF(O{tr}=0,0,E{tr + 2}/O{tr})").number_format = "0.0%"
notes = [
    "Formulas:",
    "Safety Stock = ROUNDUP(Z x Forecast Error RMSE x SQRT(Lead Time in months))",
    "Reorder Point = ROUNDUP(Avg Monthly Forecast x Lead Time in months + Safety Stock)",
    "Forecast Error RMSE = root mean squared error of the selected method on the Jul-Dec 2025 backtest.",
    "Lead time months = weeks / 4.33. Unit costs and lead times are simulated assumptions.",
]
for k, n in enumerate(notes):
    c = inv.cell(row=tr + 5 + k, column=1, value=n)
    c.font = f(bold=(k == 0), italic=(k > 0), color="595959")
widths(inv, [12, 28, 16, 18, 11, 10, 10, 12, 13, 12, 12, 13, 12, 13, 14])
inv.freeze_panes = "B7"
INV_TOTAL = tr

# ---------------- Dashboard ----------------
title(dash, "Demand Forecasting & Inventory Planning - Packaging Distributor",
      "15 SKUs | 36 months of simulated sales | 5 forecasting methods | 95% service-level "
      "inventory plan")
kpis = [
    ("Forecast accuracy (1 - WAPE)", f"='Model Comparison'!D{MC_SEL}", "0.0%"),
    ("Error reduction vs naive baseline", f"='Model Comparison'!E{MC_SEL}", "0%"),
    ("Naive baseline WAPE", "='Model Comparison'!C5", "0.0%"),
    ("Selected-model WAPE", f"='Model Comparison'!C{MC_SEL}", "0.0%"),
    ("Total safety stock value", f"='Inventory Plan'!L{INV_TOTAL}", "$#,##0"),
    ("Safety stock saved vs naive", f"='Inventory Plan'!E{INV_TOTAL + 2}", "$#,##0"),
    ("Forecast units, Jan-Jun 2026", f"='Forecast 2026'!{get_column_letter(4 + len(fmonths))}{FC_LAST + 1}", "#,##0"),
]
header(dash, 4, ["Key Result", "Value"])
for i, (lab, form, fmt) in enumerate(kpis, start=5):
    a = dash.cell(row=i, column=1, value=lab)
    b = dash.cell(row=i, column=2, value=form)
    b.number_format = fmt
    b.font = f(bold=True, size=12, color=NAVY)
    a.fill = b.fill = KPI_FILL
    a.border = b.border = BOX
    a.font = f(bold=True)

dash["D3"] = "Test period: total units, actual vs forecast"
dash["D3"].font = f(bold=True, color=NAVY)
header(dash, 4, ["Month", "Actual", "Selected Forecast", "Naive Forecast"], col=4)
tmonths = sorted(test.month.unique())
for i, m in enumerate(tmonths, start=5):
    c = dash.cell(row=i, column=4, value=pd.Timestamp(m).to_pydatetime())
    c.number_format = "mmm yyyy"
    dash.cell(row=i, column=5,
              value=f'=SUMIFS({rng("D")},{rng("C")},D{i},{rng("B")},"Naive")').number_format = "#,##0"
    dash.cell(row=i, column=6,
              value=f'=SUMIFS({rng("E")},{rng("C")},D{i},{rng("H")},"Yes")').number_format = "#,##0"
    dash.cell(row=i, column=7,
              value=f'=SUMIFS({rng("E")},{rng("C")},D{i},{rng("B")},"Naive")').number_format = "#,##0"
DL = 4 + len(tmonths)
widths(dash, [34, 14, 2, 12, 12, 16, 15])

line = LineChart()
line.title = "Jul-Dec 2025: Actual vs Forecast (all SKUs)"
line.y_axis.title = "Units"
line.y_axis.numFmt = "#,##0"
line.add_data(Reference(dash, min_col=5, max_col=7, min_row=4, max_row=DL), titles_from_data=True)
line.set_categories(Reference(dash, min_col=4, min_row=5, max_row=DL))
line.y_axis.scaling.min = 35000
line.x_axis.number_format = "mmm yy"
line.height, line.width = 8, 17
colors = ["70AD47", NAVY, "C55A11"]
for s, col in zip(line.series, colors):
    s.graphicalProperties.line.solidFill = col
    s.graphicalProperties.line.width = 28000
    s.smooth = False
line.series[2].graphicalProperties.line.dashStyle = "dash"

for ch in (line, bar):
    ch.roundedCorners = False
    ch.x_axis.delete = False
    ch.y_axis.delete = False
    if ch.title is not None:
        ch.title.overlay = False
    if ch.legend is not None:
        ch.legend.overlay = False
        ch.legend.position = "b"
dash.add_chart(line, "A14")
dash.add_chart(bar, "A31")

# ---------------- Read Me ----------------
title(readme, "About this workbook")
lines = [
    ("What it is", "A demand-forecasting and inventory-planning model for 15 packaging & shipping "
                   "SKUs (boxes, tape, mailers, stretch film, labels...)."),
    ("Data", "36 months (Jan 2023 - Dec 2025) of SIMULATED monthly sales with growth trends and a "
             "Q4 holiday-shipping peak. Not real company data."),
    ("Methods", "Naive, 3-Month Moving Average, Seasonal Naive, Simple Exponential Smoothing, "
                "Holt-Winters (trend + seasonality). Built in Python (pandas/numpy)."),
    ("How the best method was chosen", "Trained on Jan 2023 - Dec 2024, scored on Jan - Jun 2025 "
                                       "(validation). The lowest-error method per SKU was kept."),
    ("How accuracy was measured", "Re-trained on Jan 2023 - Jun 2025 and forecast Jul - Dec 2025 "
                                  "(test), a period the selection never saw. See Backtest."),
    ("Inventory plan", "Safety stock and reorder points at a 95% service level using each "
                       "method's forecast error and supplier lead time. Change the service level "
                       "in Inventory Plan!B3."),
    ("Colors", "Blue text = values/inputs from Python or assumptions. Black = Excel formulas. "
               "Yellow cell = input you can change."),
    ("Sheets", "Dashboard > Inventory Plan > Forecast 2026 > Model Comparison > Backtest > "
               "Sales by SKU (SUMIFS) > Sales History (raw data)."),
]
for i, (k, v) in enumerate(lines, start=3):
    a = readme.cell(row=i, column=1, value=k)
    a.font = f(bold=True)
    a.alignment = Alignment(vertical="top")
    b = readme.cell(row=i, column=2, value=v)
    b.font = BLACK
    b.alignment = Alignment(wrap_text=True, vertical="top")
    readme.row_dimensions[i].height = 32
widths(readme, [30, 100])

for ws in wb.worksheets:
    for row in ws.iter_rows():
        for c in row:
            if c.value is not None and c.font.name != FONT:
                c.font = Font(name=FONT, bold=c.font.bold, italic=c.font.italic,
                              color=c.font.color, size=c.font.size)
    ws.sheet_view.showGridLines = ws.title not in ("Dashboard", "Read Me")

out = "outputs/Demand_Forecasting_Inventory_Plan.xlsx"
wb.save(out)
print("Saved", out)